In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("riccardoriccio/real-time-exercise-recognition-dataset")

print("Path to dataset files:", path)

Mounting files to /kaggle/input/datasets/riccardoriccio/real-time-exercise-recognition-dataset...
Path to dataset files: /kaggle/input/datasets/riccardoriccio/real-time-exercise-recognition-dataset


In [1]:
from pathlib import Path
datasetpath = Path("/kaggle/input/datasets/riccardoriccio/real-time-exercise-recognition-dataset/final_kaggle_with_additional_video")

for exer in datasetpath.iterdir():
    if exer.is_dir():
        num_files = len([f for f in exer.iterdir() if f.is_file()])
        print(exer.name, num_files)

hammer curl 19
squat 25
push-up 25
barbell biceps curl 25
shoulder press 25


In [48]:
!pip install mediapipe==0.10.33

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 60.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 10.6 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [49]:
import cv2
import numpy as np
import mediapipe as mp

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [51]:
model_path = '/kaggle/input/models/rehmang110/mediapipe/pytorch/default/1/pose_landmarker_full.task'

In [52]:
import numpy as np
from mediapipe.tasks.python.vision import drawing_utils
from mediapipe.tasks.python.vision import drawing_styles
from mediapipe.tasks.python import vision
def draw_landmarks_on_image(rgb_image, detection_result):
  pose_landmarks_list = detection_result.pose_landmarks
  annotated_image = np.copy(rgb_image)

  pose_landmark_style = drawing_styles.get_default_pose_landmarks_style()
  pose_connection_style = drawing_utils.DrawingSpec(color=(0, 255, 0), thickness=2)

  for pose_landmarks in pose_landmarks_list:
    drawing_utils.draw_landmarks(
        image=annotated_image,
        landmark_list=pose_landmarks,
        connections=vision.PoseLandmarksConnections.POSE_LANDMARKS,
        landmark_drawing_spec=pose_landmark_style,
        connection_drawing_spec=pose_connection_style)

  return annotated_image

In [57]:
import cv2
def process_video(video_path):
    BaseOptions = mp.tasks.BaseOptions
    PoseLandmarker = mp.tasks.vision.PoseLandmarker
    PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
    VisionRunningMode = mp.tasks.vision.RunningMode
    options = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=model_path),
        running_mode=VisionRunningMode.VIDEO)
    landmarker = PoseLandmarker.create_from_options(options)

    
    cap = cv2.VideoCapture(video_path)
    landmarkers= []
    if not cap.isOpened():
        print("Error: Cannot open video")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    print(f"FPS: {fps}")
    # filename = 'output_video.mp4'
    # fourcc = cv2.VideoWriter_fourcc(*'mp4v') # Codec for .mp4
    # width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    # height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    # frame_size = (width, height) 
    # out = cv2.VideoWriter(filename, fourcc, fps, frame_size)


    frame_idx = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break  
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb_frame
        )
        timestamp_ms = int((frame_idx / fps) * 1000)
        pose_landmarker_result = landmarker.detect_for_video(mp_image, timestamp_ms)
        # anno_frame=draw_landmarks_on_image(rgb_frame ,pose_landmarker_result)
        # out.write(anno_frame)
        frame_idx += 1
        landmarkers.append(pose_landmarker_result)
        
    cap.release()
    # out.release()
    print("Processing complete.")
    # print(f"video saved as well at {filename}")
    return landmarkers



# DATASET MAKING

In [7]:
# Run
results=process_video("/kaggle/input/datasets/riccardoriccio/real-time-exercise-recognition-dataset/final_kaggle_with_additional_video/push-up/push-up_13.mp4")

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1778593835.879473     133 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778593836.012900     133 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


FPS: 29.97002997002997


W0000 00:00:1778593836.233720     133 landmark_projection_calculator.cc:78] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


Processing complete.


In [8]:
frame1= results[0]
frame1.pose_world_landmarks[0][0].x

0.03041188046336174

In [60]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

# -------------------------------
# MediaPipe landmark indices
# -------------------------------
LANDMARKS = {
    "left_shoulder": 11,
    "right_shoulder": 12,
    "left_elbow": 13,
    "right_elbow": 14,
    "left_wrist": 15,
    "right_wrist": 16,
    "left_hip": 23,
    "right_hip": 24,
    "left_knee": 25,
    "right_knee": 26,
}

# -------------------------------
# Distance function (3D)
# -------------------------------
def distance_3d(a, b):
    return np.sqrt((a.x - b.x)**2 + (a.y - b.y)**2 + (a.z - b.z)**2)


# -------------------------------
# Signal extraction per exercise
# -------------------------------
def compute_signal(results, exercise="pushup"):

    signal = []

    for frame in results:
        if not frame.pose_world_landmarks:
            continue

        lm = frame.pose_world_landmarks[0]

        # LEFT SIDE
        shoulder = lm[LANDMARKS["left_shoulder"]]
        elbow = lm[LANDMARKS["left_elbow"]]
        wrist = lm[LANDMARKS["left_wrist"]]
        hip = lm[LANDMARKS["left_hip"]]
        knee = lm[LANDMARKS["left_knee"]]

        if exercise == "pushup":
            value = distance_3d(shoulder, wrist)

        elif exercise == "squat":
            value = distance_3d(hip, knee)

        elif exercise == "bicep_curl":
            value = distance_3d(shoulder, wrist)

        elif exercise == "shoulder_press":
            value = wrist.y - shoulder.y

        else:
            raise ValueError("Unknown exercise")

        signal.append(value)

    return np.array(signal)


# -------------------------------
# Smooth signal (stronger)
# -------------------------------
def smooth_signal(signal, window=11):
    return np.convolve(signal, np.ones(window)/window, mode='same')


# -------------------------------
# Normalize signal
# -------------------------------
def normalize_signal(signal):
    return (signal - np.min(signal)) / (np.max(signal) - np.min(signal) + 1e-6)


# -------------------------------
# Peak + valley detection + plot
# -------------------------------
def analyze_and_plot(results, exercise="pushup"):

    # 1. Compute signal
    signal = compute_signal(results, exercise)

    # 2. Smooth
    signal = smooth_signal(signal, window=11)

    # 3. Normalize
    signal = normalize_signal(signal)

    # 4. Dynamic distance (depends on length)
    min_distance = max(10, len(signal) // 20)

    # 5. Detect ONLY valleys (robust)
    valleys, _ = find_peaks(
        -signal,
        distance=min_distance,
        prominence=0.2
    )

    # (Optional) peaks just for visualization
    peaks, _ = find_peaks(
        signal,
        distance=min_distance,
        prominence=0.2
    )

    # # -------------------------------
    # # Plot
    # # -------------------------------
    # plt.figure(figsize=(12, 5))
    # plt.plot(signal, label="Signal", linewidth=2)

    # plt.scatter(valleys, signal[valleys], color='green', s=60, label="Valleys (Reps)")
    # plt.scatter(peaks, signal[peaks], color='red', s=40, alpha=0.5, label="Peaks (debug)")

    # plt.title(f"{exercise.upper()} Signal (Robust Detection)")
    # plt.xlabel("Frame")
    # plt.ylabel("Normalized Signal")
    # plt.legend()
    # plt.grid()

    # plt.show()

    # # -------------------------------
    # # Rep count
    # # -------------------------------
    reps = len(valleys)
    # print(f"Detected reps: {reps}")

    return reps,valleys,peaks

In [63]:
def segment_exact_reps(results,  peaks, valleys):

    segmented_reps = []

    for valley in valleys:

        # -----------------------------
        # Find previous peak
        # -----------------------------
        prev_peaks = peaks[peaks < valley]

        if len(prev_peaks) == 0:
            continue

        start = prev_peaks[-1]

        # -----------------------------
        # Find next peak
        # -----------------------------
        next_peaks = peaks[peaks > valley]

        if len(next_peaks) == 0:
            continue

        end = next_peaks[0]

        # -----------------------------
        # Extract rep
        # -----------------------------
        rep_sequence = results[start:end]

        segmented_reps.append({
            "start": start,
            "valley": valley,
            "end": end,
            "frames": rep_sequence
        })

    return segmented_reps

In [11]:
reps, valleys, peaks = analyze_and_plot(results, "pushup")

segmented = segment_exact_reps(
    results,
    peaks,
    valleys
)
print(len(segmented))
print(segmented[0].keys())
print(segmented[0]['end'])
print(reps)
print(valleys)
print(peaks)

2
dict_keys(['start', 'valley', 'end', 'frames'])
82
2
[ 48 121]
[ 11  82 139]


In [29]:
import os
import numpy as np


def save_reps_dataset(
    dataset,
    exercise_name,
    save_dir
):
    os.makedirs(save_dir, exist_ok=True)

    existing_files = [
        f for f in os.listdir(save_dir)
        if f.startswith(exercise_name)
        and f.endswith(".npz")
    ]
    indices = []

    for f in existing_files:

        try:
            idx = int(
                f.replace(".npz", "").split("_")[-1]
            )

            indices.append(idx)

        except:
            pass

    if len(indices) == 0:
        start_idx = 0
    else:
        start_idx = max(indices) + 1

    for i, rep_features in enumerate(dataset):

        file_index = start_idx + i

        filename = f"{exercise_name}_{file_index}.npz"

        filepath = os.path.join(
            save_dir,
            filename
        )

        np.savez_compressed(
            filepath,
            features=rep_features,
            label=exercise_name
        )


In [64]:
import numpy as np
from scipy.interpolate import interp1d

# =========================================================
# LANDMARK INDICES
# =========================================================

LANDMARKS = {
    "left_shoulder": 11,
    "right_shoulder": 12,
    "left_elbow": 13,
    "right_elbow": 14,
    "left_wrist": 15,
    "right_wrist": 16,
    "left_hip": 23,
    "right_hip": 24,
    "left_knee": 25,
    "right_knee": 26,
    "left_ankle": 27,
    "right_ankle": 28,
}

# =========================================================
# BASIC UTILITIES
# =========================================================

def point_to_array(p):
    return np.array([p.x, p.y, p.z])


# =========================================================
# ANGLE CALCULATION
# =========================================================

def calculate_angle(a, b, c):

    a = point_to_array(a)
    b = point_to_array(b)
    c = point_to_array(c)

    ba = a - b
    bc = c - b

    cosine = np.dot(ba, bc) / (
        np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6
    )

    angle = np.degrees(
        np.arccos(
            np.clip(cosine, -1.0, 1.0)
        )
    )

    return angle


# =========================================================
# NORMALIZE LANDMARKS
# =========================================================

def normalize_landmarks(lm):

    # ------------------------------------
    # Convert to numpy array
    # ------------------------------------
    points = np.array([
        [p.x, p.y, p.z]
        for p in lm
    ])

    # ------------------------------------
    # HIP CENTER
    # ------------------------------------
    left_hip = points[23]
    right_hip = points[24]

    hip_center = (left_hip + right_hip) / 2

    # Translation normalization
    points = points - hip_center

    # ------------------------------------
    # SCALE NORMALIZATION
    # ------------------------------------
    left_shoulder = points[11]
    right_shoulder = points[12]

    shoulder_width = np.linalg.norm(
        left_shoulder - right_shoulder
    )

    torso_height = np.linalg.norm(
        ((points[11] + points[12]) / 2)
    )

    scale = max(
        shoulder_width,
        torso_height,
        1e-6
    )

    points = points / scale

    return points


# =========================================================
# EXTRACT FEATURES FROM ONE FRAME
# =========================================================

def extract_frame_features(lm):

    # ------------------------------------
    # NORMALIZED XYZ
    # ------------------------------------
    normalized_points = normalize_landmarks(lm)

    xyz_features = normalized_points.flatten()

    # ------------------------------------
    # ANGLES
    # ------------------------------------
    angles = []

    # LEFT ELBOW
    angles.append(
        calculate_angle(
            lm[11], lm[13], lm[15]
        )
    )

    # RIGHT ELBOW
    angles.append(
        calculate_angle(
            lm[12], lm[14], lm[16]
        )
    )

    # LEFT SHOULDER
    angles.append(
        calculate_angle(
            lm[13], lm[11], lm[23]
        )
    )

    # RIGHT SHOULDER
    angles.append(
        calculate_angle(
            lm[14], lm[12], lm[24]
        )
    )

    # LEFT KNEE
    angles.append(
        calculate_angle(
            lm[23], lm[25], lm[27]
        )
    )

    # RIGHT KNEE
    angles.append(
        calculate_angle(
            lm[24], lm[26], lm[28]
        )
    )

    # LEFT HIP
    angles.append(
        calculate_angle(
            lm[11], lm[23], lm[25]
        )
    )

    # RIGHT HIP
    angles.append(
        calculate_angle(
            lm[12], lm[24], lm[26]
        )
    )

    # ------------------------------------
    # COMBINE FEATURES
    # ------------------------------------
    features = np.concatenate([
        xyz_features,
        np.array(angles)
    ])

    return features


# =========================================================
# RESAMPLE SEQUENCE TO FIXED LENGTH
# =========================================================

def resample_sequence(sequence, target_length=40):

    old_length = len(sequence)

    if old_length < 2:
        return None

    x_old = np.linspace(0, 1, old_length)
    x_new = np.linspace(0, 1, target_length)

    interpolator = interp1d(
        x_old,
        sequence,
        axis=0
    )

    new_sequence = interpolator(x_new)

    return new_sequence


# =========================================================
# PROCESS ONE REP
# =========================================================

def process_single_rep(rep_data, target_frames=40):

    """
    rep_data:
        one segmented rep dictionary
    """

    frames = rep_data["frames"]

    all_features = []

    for frame in frames:

        if not frame.pose_world_landmarks:
            continue

        lm = frame.pose_world_landmarks[0]

        # Extract frame features
        features = extract_frame_features(lm)

        all_features.append(features)

    all_features = np.array(all_features)

    # ------------------------------------
    # TEMPORAL NORMALIZATION
    # ------------------------------------
    all_features = resample_sequence(
        all_features,
        target_length=target_frames
    )

    return all_features


# =========================================================
# PROCESS ALL REPS
# =========================================================

def process_all_segmented_reps(segmented_reps):

    dataset = []

    for rep in segmented_reps:

        processed_rep = process_single_rep(rep)

        if processed_rep is not None:

            dataset.append(processed_rep)

    dataset = np.array(dataset)

    return dataset

In [30]:
results=process_video("/kaggle/input/datasets/riccardoriccio/real-time-exercise-recognition-dataset/final_kaggle_with_additional_video/push-up/push-up_13.mp4")
reps, valleys, peaks = analyze_and_plot(results, "pushup")
segmented = segment_exact_reps(
    results,
    peaks,
    valleys
)
dataset = process_all_segmented_reps(segmented)
save_reps_dataset(
    dataset=dataset,
    exercise_name="pushup",
    save_dir="my_dataset/pushup"
)
print(dataset.shape)

W0000 00:00:1778596789.750622     214 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778596789.780084     214 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


FPS: 29.97002997002997
Processing complete.
(2, 40, 107)


In [31]:
data = np.load("/kaggle/working/my_dataset/pushup/pushup_2.npz")

features = data["features"]
label = data["label"]

print(features.shape)
print(label)

(40, 107)
pushup


In [43]:
from pathlib import Path
from tqdm import tqdm

dataset_path = Path(
    "/kaggle/input/datasets/riccardoriccio/real-time-exercise-recognition-dataset/final_kaggle_with_additional_video"
)

for exercise in dataset_path.iterdir():

    exercise_name = exercise.name

    if exercise_name == "barbell biceps curl":
        exercise_name = "bicep_curl"

    if exercise_name == "shoulder press":
        exercise_name = "shoulder_press"

    if exercise_name == "push-up":
        exercise_name = "pushup"

    if exercise_name == "hammer curl":
        continue

    print(f"\nProcessing Exercise: {exercise_name}")

    video_files = list(exercise.iterdir())

    for files in tqdm(video_files, desc=exercise_name):

        results = process_video(str(files))

        reps, valleys, peaks = analyze_and_plot(
            results,
            exercise_name
        )

        segmented = segment_exact_reps(
            results,
            peaks,
            valleys
        )

        dataset = process_all_segmented_reps(
            segmented
        )

        save_reps_dataset(
            dataset=dataset,
            exercise_name=exercise_name,
            save_dir=f"my_dataset/{exercise_name}"
        )

        print(f"Processed: {files.name}")
        print(dataset.shape)


Processing Exercise: squat


squat:   0%|          | 0/25 [00:00<?, ?it/s]W0000 00:00:1778598589.397774     646 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778598589.452892     649 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


FPS: 29.97420060980377


squat:   4%|▍         | 1/25 [00:29<11:38, 29.09s/it]

Processing complete.
Processed: squat_3.MOV
(6, 40, 107)


...

Processing complete.
Processed: shoulder press_8.mp4
(0,)
FPS: 29.97002997002997


W0000 00:00:1778599507.450835    1824 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778599507.473035    1824 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
shoulder_press:  96%|█████████▌| 24/25 [05:16<00:09,  9.24s/it]

Processing complete.
Processed: shoulder press_11.mp4
(2, 40, 107)
FPS: 25.0


W0000 00:00:1778599515.627998    1836 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778599515.648593    1836 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
shoulder_press: 100%|██████████| 25/25 [05:35<00:00, 13.40s/it]

Processing complete.
Processed: shoulder press_24.mp4
(9, 40, 107)


In [44]:
import shutil

# Folder to zip
folder_to_zip = "/kaggle/working/my_dataset"

# Output zip name
output_zip = "my_dataset.zip"

# Create zip
shutil.make_archive(
    base_name="my_dataset",
    format="zip",
    root_dir=folder_to_zip
)

print(f"Dataset zipped successfully: {output_zip}")

Dataset zipped successfully: my_dataset.zip


# Dataset Preparation

In [1]:
import os
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
def load_dataset(dataset_dir):

    X = []
    y = []

    dataset_dir = Path(dataset_dir)
    for exercise_folder in dataset_dir.iterdir():

        if not exercise_folder.is_dir():
            continue

        exercise_name = exercise_folder.name

        print(f"Loading: {exercise_name}")

        # -
        for sample_file in exercise_folder.glob("*.npz"):

            data = np.load(sample_file)

            features = data["features"]
            label = data["label"]

            X.append(features)
            y.append(label)

    X = np.array(X)
    y = np.array(y)

    return X, y

def prepare_gru_dataset(
    dataset_dir,
    test_size=0.2,
    random_state=42
):

    X, y = load_dataset(dataset_dir)

    print("\nRaw Dataset Shape")
    print("X:", X.shape)
    print("y:", y.shape)

 
    # ----------------------------------------
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    y_categorical = to_categorical(y_encoded)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y_categorical,
        test_size=test_size,
        random_state=random_state,
        stratify=y_encoded
    )

    print("\nTrain/Test Shapes")
    print("X_train:", X_train.shape)
    print("X_test :", X_test.shape)
    print("y_train:", y_train.shape)
    print("y_test :", y_test.shape)

    return (
        X_train,
        X_test,
        y_train,
        y_test,
        label_encoder
    )

2026-05-16 08:32:12.970073: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778920333.154192      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778920333.205757      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778920333.644687      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778920333.644735      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778920333.644738      57 computation_placer.cc:177] computation placer alr

In [2]:
X_train, X_test, y_train, y_test, label_encoder = prepare_gru_dataset(
    dataset_dir="/kaggle/input/datasets/rehmang110/exercises-dataset"
)

Loading: pushup
Loading: squat
Loading: bicep_curl
Loading: shoulder_press

Raw Dataset Shape
X: (264, 40, 107)
y: (264,)

Train/Test Shapes
X_train: (211, 40, 107)
X_test : (53, 40, 107)
y_train: (211, 4)
y_test : (53, 4)


# MODEL MAKING

In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (GRU, 
Dense ,Dropout,BatchNormalization)

from tensorflow.keras.callbacks import (
EarlyStopping,
ReduceLROnPlateau
)
def build_gru_model(input_shape, num_classes):

    model = Sequential([

        GRU(
            128,
            return_sequences=True,
            input_shape=input_shape
        ),

        BatchNormalization(),

        Dropout(0.3),

        GRU(
            64,
            return_sequences=False
        ),

        BatchNormalization(),

        Dropout(0.3),

        
        Dense(
            64,
            activation='relu'
        ),

        Dropout(0.3),

        Dense(
            num_classes,
            activation='softmax'
        )
    ])

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model



In [37]:
input_shape = (
    X_train.shape[1],   # 40 frames
    X_train.shape[2]    # feature dimension 107
)

num_classes = y_train.shape[1]
model = build_gru_model(
    input_shape,
    num_classes
)

In [42]:
model.summary()

Model: "sequential_14"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_28 (GRU)                    │ (None, 40, 128)        │        91,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_28          │ (None, 40, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_42 (Dropout)            │ (None, 40, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_29 (GRU)                    │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_29          │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_43 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_44 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 133,444 (521.27 KB)

 Trainable params: 133,060 (519.77 KB)

 Non-trainable params: 384 (1.50 KB)

In [43]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5
)

In [45]:
history = model.fit(

    X_train,
    y_train,

    validation_data=(
        X_test,
        y_test
    ),

    epochs=100,

    batch_size=32,

    callbacks=[
        early_stopping,
        reduce_lr
    ]
)

Epoch 1/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 72ms/step - accuracy: 0.9563 - loss: 0.1091 - val_accuracy: 0.7547 - val_loss: 0.5805 - learning_rate: 0.0010
Epoch 2/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.9786 - loss: 0.0711 - val_accuracy: 0.8491 - val_loss: 0.4810 - learning_rate: 0.0010
Epoch 3/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - accuracy: 0.9876 - loss: 0.0687 - val_accuracy: 0.8679 - val_loss: 0.3830 - learning_rate: 0.0010
Epoch 4/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - accuracy: 0.9863 - loss: 0.0559 - val_accuracy: 0.8679 - val_loss: 0.3507 - learning_rate: 0.0010
Epoch 5/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 72ms/step - accuracy: 0.9828 - loss: 0.0537 - val_accuracy: 0.9057 - val_loss: 0.3360 - learning_rate: 0.0010
Epoch 6/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 70ms/step - accuracy: 0.9970 - loss: 0.0410 - val_accuracy: 0.9245 - val_loss: 0.2793 - learning_rate: 0.0010
Epoch 7/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.9872 - loss: 0.0488 - val_accuracy: 

In [66]:
video_path = "/kaggle/input/datasets/rehmang110/aesyhe/6893306-uhd_3840_2160_25fps.mp4"

results = process_video(video_path)

reps, valleys, peaks = analyze_and_plot(
    results,
    "pushup"
)

segmented = segment_exact_reps(
    results,
    peaks,
    valleys
)

dataset = process_all_segmented_reps(
    segmented
)

for i, rep_features in enumerate(dataset):

    sample = np.expand_dims(rep_features, axis=0)

    pred = model.predict(sample, verbose=0)

    class_idx = np.argmax(pred)

    confidence = pred[0][class_idx]

    class_name = label_encoder.inverse_transform(
        [class_idx]
    )[0]

    print(
        f"Rep {i+1}: "
        f"{class_name} "
        f"({confidence:.2f})"
    )

W0000 00:00:1778922840.265525    1328 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778922840.296356    1329 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


FPS: 25.0


W0000 00:00:1778922840.736229    1328 landmark_projection_calculator.cc:78] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


Processing complete.
Rep 1: squat (0.91)
Rep 2: squat (0.96)
Rep 3: pushup (0.99)
